## Objective

This notebook focuses on converting raw UE-level 5G traffic data
into a labeled and preprocessed dataset suitable for machine
learning.

Specifically, we:
- Derive attack labels using known DDoS attack time windows
- Clean and select relevant features
- Handle missing values
- Prepare the dataset for ML model training

This process follows the methodology described in the
reference IEEE paper.


In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler


In [2]:
DATA_PATH = "C:/Users/saura/cn-project-network-security/data/amari_ue_data_classic_tabular.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

# Convert timestamp
df["_time"] = pd.to_datetime(df["_time"], errors="coerce")


In [3]:
df.shape


(424660, 80)

In [4]:
# Drop rows with invalid timestamps
df = df.dropna(subset=["_time"])

df["_time"].min(), df["_time"].max()


(Timestamp('2024-08-17 12:00:01.700000+0000', tz='UTC'),
 Timestamp('2024-08-22 06:59:55.404000+0000', tz='UTC'))

## Attack Time Windows

Based on the dataset documentation and reference paper,
DDoS attacks were launched during predefined time intervals.
Samples falling within these windows are labeled as attacks.


In [7]:
attack_windows = [
    ("2024-08-18 07:00:00", "2024-08-18 08:00:00"),  # SYN Flood
    ("2024-08-19 07:00:00", "2024-08-19 09:41:00"),  # ICMP Flood
    ("2024-08-19 17:00:00", "2024-08-19 18:00:00"),  # UDP Fragmentation
    ("2024-08-21 12:00:00", "2024-08-21 13:00:00"),  # DNS Flood
    ("2024-08-21 17:00:00", "2024-08-21 18:00:00")   # GTP-U Flood
]

# Convert attack windows to timezone-aware UTC timestamps
attack_windows = [
    (
        pd.to_datetime(start).tz_localize("UTC"),
        pd.to_datetime(end).tz_localize("UTC")
    )
    for start, end in attack_windows
]


In [8]:
df["attack_label"] = 0

for start, end in attack_windows:
    df.loc[
        (df["_time"] >= start) & (df["_time"] <= end),
        "attack_label"
    ] = 1


In [9]:
df["attack_label"].value_counts()


attack_label
0    397597
1     26624
Name: count, dtype: int64

In [10]:
df["attack_label"].value_counts(normalize=True) * 100


attack_label
0    93.724026
1     6.275974
Name: proportion, dtype: float64

In [11]:
df.groupby(df["_time"].dt.date)["attack_label"].mean()


_time
2024-08-17    0.000000
2024-08-18    0.041820
2024-08-19    0.153407
2024-08-20    0.000000
2024-08-21    0.083311
2024-08-22    0.000000
Name: attack_label, dtype: float64

## Feature Selection: Removing Non-ML Columns

Certain columns in the dataset correspond to identifiers,
addresses, or configuration parameters. These features do
not represent network behavior and may introduce bias or
leakage if used for model training.

Such columns are removed prior to preprocessing.


In [13]:
non_ml_cols = [
    "_time",            # timestamp (used only for labeling)
    "imeisv",           # UE identifier
    "5g_tmsi",
    "amf_ue_id",
    "rnti",
    "ran_id",
    "ran_plmn",
    "tac",
    "tac_plmn",
    "registered",
    
    # IP / addressing fields
    "bearer_0_ip",
    "bearer_0_ipv6",
    "bearer_1_ip",
    "bearer_1_ipv6",
]

# Drop only columns that exist
df_ml = df.drop(columns=[c for c in non_ml_cols if c in df.columns])

df_ml.shape


(424221, 67)

In [14]:
df_ml.columns


Index(['bearer_0_apn', 'bearer_0_dl_total_bytes', 'bearer_0_pdu_session_id',
       'bearer_0_qos_flow_id', 'bearer_0_sst', 'bearer_0_ul_total_bytes',
       'bearer_1_apn', 'bearer_1_dl_total_bytes', 'bearer_1_pdu_session_id',
       'bearer_1_qos_flow_id', 'bearer_1_sst', 'bearer_1_ul_total_bytes',
       'cell_1_cell_id', 'cell_1_cqi', 'cell_1_dl_bitrate', 'cell_1_dl_err',
       'cell_1_dl_mcs', 'cell_1_dl_retx', 'cell_1_dl_tx', 'cell_1_epre',
       'cell_1_initial_ta', 'cell_1_p_ue', 'cell_1_pusch_snr', 'cell_1_ri',
       'cell_1_turbo_decoder_avg', 'cell_1_turbo_decoder_max',
       'cell_1_turbo_decoder_min', 'cell_1_ul_bitrate', 'cell_1_ul_err',
       'cell_1_ul_mcs', 'cell_1_ul_n_layer', 'cell_1_ul_path_loss',
       'cell_1_ul_phr', 'cell_1_ul_rank', 'cell_1_ul_retx', 'cell_1_ul_tx',
       'dl_bitrate', 'ran_ue_id', 't3512', 'ue_aggregate_max_bitrate_dl',
       'ue_aggregate_max_bitrate_ul', 'ul_bitrate', 'cell_3_cell_id',
       'cell_3_cqi', 'cell_3_dl_bitrate', 'cell_

After removing identifiers and addressing information,
the remaining features primarily capture traffic volume,
radio conditions, retransmissions, and bearer-level activity.
These features reflect UE behavior and are suitable for
ML-based anomaly detection.


## Consolidation of Cell-wise Metrics

The dataset provides radio and retransmission metrics
separately for each cell (e.g., cell_1, cell_3).
However, a UE is associated with a single serving cell
at any given time.

To obtain UE-level features consistent with the
methodology of the reference paper, cell-wise metrics
are consolidated by selecting the maximum value across
all available cells.


In [15]:
# Identify uplink retransmission columns
ul_retx_cols = [
    col for col in df_ml.columns
    if col.startswith("cell_") and "ul_retx" in col
]

ul_retx_cols


['cell_1_ul_retx', 'cell_3_ul_retx']

In [16]:
df_ml["ul_retx_max"] = df_ml[ul_retx_cols].max(axis=1)


In [17]:
# Identify downlink retransmission columns
dl_retx_cols = [
    col for col in df_ml.columns
    if col.startswith("cell_") and "dl_retx" in col
]

dl_retx_cols


['cell_1_dl_retx', 'cell_3_dl_retx']

In [18]:
df_ml["dl_retx_max"] = df_ml[dl_retx_cols].max(axis=1)


In [19]:
df_ml = df_ml.drop(columns=ul_retx_cols + dl_retx_cols)

df_ml.shape


(424221, 65)

In [20]:
df_ml[["ul_retx_max", "dl_retx_max"]].describe()


,ul_retx_max,dl_retx_max
count,424221.000000,424221.000000
mean,42.917288,35.126116
std,82.109536,61.886528
min,0.000000,0.000000
25%,0.000000,2.000000
50%,1.000000,5.000000
75%,24.000000,19.000000
max,656.000000,770.000000


Cell-wise retransmission metrics were consolidated into
UE-level features by taking the maximum value across cells.
This transformation preserves the behavior of the serving
cell while reducing dimensionality and redundancy.

The resulting features are more suitable for supervised
learning and align with the abstraction used in the
reference paper.


In [21]:
# Percentage of missing values per column
missing_percent = (df_ml.isna().sum() / len(df_ml)) * 100

missing_percent.sort_values(ascending=False).head(10)


cell_3_ul_n_layer           56.890394
cell_3_ul_mcs               56.890394
cell_3_turbo_decoder_max    56.890159
cell_3_turbo_decoder_avg    56.890159
cell_3_turbo_decoder_min    56.890159
cell_3_ul_phr               56.873186
cell_3_ul_path_loss         56.873186
cell_3_p_ue                 56.873186
cell_3_dl_mcs               56.873186
cell_3_cell_id              56.872951
dtype: float64

## Handling Missing Values

Missing values are expected in real-world 5G network datasets
due to intermittent UE activity, cell transitions, and
measurement granularity.

Since the majority of features are numerical traffic and
radio metrics, missing values are handled using median
imputation, which is robust to skewed and heavy-tailed
distributions commonly observed in network traffic.


In [22]:
# Fill missing numerical values with median
df_ml = df_ml.fillna(df_ml.median(numeric_only=True))


In [23]:
df_ml.isna().sum().sum()


76188

In [24]:
# Check remaining NaNs per column
df_ml.isna().sum().sort_values(ascending=False).head(10)


bearer_1_apn         76188
bearer_0_apn             0
cell_3_cqi               0
cell_3_initial_ta        0
cell_3_epre              0
cell_3_dl_tx             0
cell_3_dl_mcs            0
cell_3_dl_err            0
cell_3_dl_bitrate        0
cell_3_cell_id           0
dtype: int64

In [25]:
# Drop APN (categorical, non-behavioral) columns
apn_cols = [col for col in df_ml.columns if col.endswith("_apn")]

apn_cols


['bearer_0_apn', 'bearer_1_apn']

In [26]:
df_ml = df_ml.drop(columns=apn_cols)


In [27]:
df_ml.isna().sum().sum()


0

Bearer APN fields were removed from the feature set, as they
represent configuration-level categorical information rather
than UE traffic behavior. Additionally, some APN fields were
entirely missing due to inactive secondary bearers.

Removing these fields ensures that the final dataset contains
only numerical, behavior-driven features suitable for ML.


## Feature Scaling

In [28]:
from sklearn.preprocessing import StandardScaler


In [29]:
# Separate features and target
X = df_ml.drop(columns=["attack_label"])
y = df_ml["attack_label"]

X.shape, y.shape


((424221, 62), (424221,))

In [30]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


In [31]:
df_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns,
    index=df_ml.index
)

df_scaled["attack_label"] = y


In [32]:
df_scaled.shape


(424221, 63)

In [33]:
df_scaled.describe().T.head()


,count,mean,std,min,25%,50%,75%,max
bearer_0_dl_total_bytes,424221.0,3.323072e-17,1.000001,-0.376695,-0.376328,-0.376077,-0.316758,5.016307
bearer_0_pdu_session_id,424221.0,1.029080e-16,1.000001,-0.678566,-0.678566,-0.678566,0.748617,2.175800
bearer_0_qos_flow_id,424221.0,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
bearer_0_sst,424221.0,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
bearer_0_ul_total_bytes,424221.0,-8.575669e-18,1.000001,-0.362124,-0.362116,-0.362110,-0.306293,4.548620


In [34]:
df_scaled["attack_label"].value_counts()


attack_label
0    397597
1     26624
Name: count, dtype: int64

In [35]:
# Remove constant (zero-variance) features
stds = df_scaled.drop(columns=["attack_label"]).std()
zero_var_cols = stds[stds == 0].index.tolist()

zero_var_cols


['bearer_0_qos_flow_id',
 'bearer_0_sst',
 'bearer_1_qos_flow_id',
 'bearer_1_sst',
 'cell_1_cell_id',
 'cell_1_ul_n_layer',
 'cell_1_ul_rank',
 't3512',
 'ue_aggregate_max_bitrate_dl',
 'ue_aggregate_max_bitrate_ul',
 'cell_3_cell_id',
 'cell_3_ul_rank',
 'cell_3_ul_n_layer']

In [36]:
df_scaled = df_scaled.drop(columns=zero_var_cols)


In [37]:
df_scaled.shape


(424221, 50)

Features with zero variance across all samples were removed,
as they do not contribute to discrimination between benign
and attack traffic and may negatively impact model training.


In [39]:
FINAL_DATA_PATH = "../data/processed/ue_attack_labeled_scaled.csv"

df_scaled.to_csv(FINAL_DATA_PATH, index=False)

print(f"Saved final dataset to {FINAL_DATA_PATH}")


Saved final dataset to ../data/processed/ue_attack_labeled_scaled.csv


In [40]:
import joblib

SCALER_PATH = "../data/processed/standard_scaler.pkl"

joblib.dump(scaler, SCALER_PATH)

print(f"Saved scaler to {SCALER_PATH}")


Saved scaler to ../data/processed/standard_scaler.pkl


## Summary of Attack Labeling and Preprocessing

In this notebook, raw UE-level 5G network measurements were
transformed into a clean, labeled, and machine-learning-ready
dataset, following the methodology described in the reference
IEEE paper.

The key steps performed are summarized below:

1. **Attack Labeling**
   - DDoS attack labels were derived using predefined time
     windows corresponding to known attack periods.
   - All samples falling within these windows were labeled
     as malicious, while the remaining samples were labeled
     as benign.
   - The resulting dataset exhibits strong class imbalance
     (~6% attack traffic), reflecting realistic network
     security conditions.

2. **Feature Selection**
   - Non-behavioral and identifying attributes such as UE IDs,
     IP addresses, and configuration parameters were removed
     to prevent information leakage and overfitting.
   - Only traffic-, radio-, and bearer-level numerical features
     were retained for modeling.

3. **Consolidation of Cell-wise Metrics**
   - Cell-specific retransmission counters were consolidated
     into UE-level features by selecting the maximum value
     across available cells.
   - This transformation aligns the raw dataset with the
     abstraction used in the reference paper and reduces
     feature redundancy.

4. **Handling Missing Values**
   - Missing values, arising from UE inactivity and reporting
     behavior, were handled using median imputation.
   - Categorical and configuration fields with pervasive
     missingness (e.g., APN fields) were excluded from the
     feature set.

5. **Feature Scaling**
   - All numerical features were standardized using z-score
     normalization to ensure comparable scales across metrics.
   - Zero-variance features were identified and removed, as
     they do not contribute to discriminative learning.

6. **Data Persistence**
   - The final preprocessed dataset was saved to disk and a
     reusable data-loading function was defined to enable
     consistent reuse across subsequent notebooks without
     recomputing preprocessing steps.

At the end of this notebook, the dataset is fully labeled,
numerically stable, and ready for machine learning model
training.
